#### Exploración de los Datos

In [13]:
import numpy as np
import pandas as pd

df = pd.read_csv("Crime_Population.csv", encoding="utf-8")

df.head(15)

,delito,bien_afectado,colonia,municipio,año,mes,dia,hora,min,hora_conocida,poblacion_colonia,poblacion_femenina,poblacion_masculina
0,Abuso sexual infantil,La libertad y la seguridad sexual,JARDINES DEL BOSQUE,GUADALAJARA,2020,3,8,0.0,0.0,True,6068.0,3203.0,2843.0
1,Abuso sexual infantil,La libertad y la seguridad sexual,BALCONES DE SANTA CRUZ,TONALA,2020,3,11,0.0,0.0,True,225.0,118.0,107.0
2,Abuso sexual infantil,La libertad y la seguridad sexual,EL BETHEL,GUADALAJARA,2020,3,14,0.0,0.0,True,8987.0,4561.0,4426.0
3,Abuso sexual infantil,La libertad y la seguridad sexual,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,2020,3,14,0.0,0.0,True,NaN,NaN,NaN
4,Abuso sexual infantil,La libertad y la seguridad sexual,MIRAVALLE,SAN PEDRO TLAQUEPAQUE,2020,3,16,0.0,0.0,True,24199.0,12673.0,11522.0
5,Abuso sexual infantil,La libertad y la seguridad sexual,SANTA MARIA,GUADALAJARA,2020,1,14,0.0,0.0,True,10550.0,5413.0,5137.0
6,Abuso sexual infantil,La libertad y la seguridad sexual,SAN ESTEBAN,ZAPOPAN,2020,9,2,0.0,0.0,True,1736.0,866.0,870.0
7,Abuso sexual infantil,La libertad y la seguridad sexual,SAN PEDRITO,SAN PEDRO TLAQUEPAQUE,2020,1,25,0.0,0.0,True,14689.0,7419.0,7252.0
8,Abuso sexual infantil,La libertad y la seguridad sexual,MODERNA,GUADALAJARA,2020,11,5,0.0,0.0,True,4015.0,2037.0,1894.0
9,Abuso sexual infantil,La libertad y la seguridad sexual,LOMAS DE POLANCO,GUADALAJARA,2020,1,4,0.0,0.0,True,21615.0,11165.0,10446.0


Se puede observar que las colonias que no poseen información sobre el censo poblacional del INEGI, son consideradas como muy peligrosas.

In [11]:
# convertimos valor booleano a binario
df["hora_conocida"] = df["hora_conocida"].astype(int)

df["hora_conocida"].value_counts()

hora_conocida
1    275878
0     30264
Name: count, dtype: int64

In [12]:
# sustituimos dia por dia_semana y es_fin_semana
fecha = pd.to_datetime(df[["año", "mes", "dia"]].rename(
    columns={"año": "year", "mes": "month", "dia": "day"}))

df["dia_semana"] = fecha.dt.dayofweek            # 0 = lunes ... 6 = domingo
df["es_fin_semana"] = (df["dia_semana"] >= 5).astype(int)

df = df.drop(columns="dia")

df.head()

,delito,bien_afectado,colonia,municipio,año,mes,hora,min,hora_conocida,poblacion_colonia,poblacion_femenina,poblacion_masculina,dia_semana,es_fin_semana
0,Abuso sexual infantil,La libertad y la seguridad sexual,JARDINES DEL BOSQUE,GUADALAJARA,2020,3,0.0,0.0,1,6068.0,3203.0,2843.0,6,1
1,Abuso sexual infantil,La libertad y la seguridad sexual,BALCONES DE SANTA CRUZ,TONALA,2020,3,0.0,0.0,1,225.0,118.0,107.0,2,0
2,Abuso sexual infantil,La libertad y la seguridad sexual,EL BETHEL,GUADALAJARA,2020,3,0.0,0.0,1,8987.0,4561.0,4426.0,5,1
3,Abuso sexual infantil,La libertad y la seguridad sexual,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,2020,3,0.0,0.0,1,NaN,NaN,NaN,5,1
4,Abuso sexual infantil,La libertad y la seguridad sexual,MIRAVALLE,SAN PEDRO TLAQUEPAQUE,2020,3,0.0,0.0,1,24199.0,12673.0,11522.0,0,0


In [4]:
# quitamos las filas sin datos de poblacion
pob = ["poblacion_colonia", "poblacion_femenina", "poblacion_masculina"]

antes = len(df)
df = df.dropna(subset=pob).reset_index(drop=True)
df = df.dropna(subset=["hora"]).reset_index(drop=True)

print(f"filas eliminadas: {antes - len(df)} ({(antes - len(df)) / antes:.1%})")
print("filas restantes:", len(df))
df["hora"].isna().sum()

filas eliminadas: 48957 (16.0%)
filas restantes: 257185


np.int64(0)

In [5]:
# las tres columnas de poblacion estan casi perfectamente colineales (r > 0.999):
# nos quedamos con el tamano y la composicion por sexo
df["razon_femenina"] = df["poblacion_femenina"] / df["poblacion_colonia"]

df = df.drop(columns=["poblacion_femenina", "poblacion_masculina"])

df[["poblacion_colonia", "razon_femenina"]].describe()

,poblacion_colonia,razon_femenina
count,257185.000000,257185.000000
mean,8444.238999,0.509948
std,7446.390999,0.029577
min,3.000000,0.000000
25%,3031.000000,0.503931
50%,6495.000000,0.510676
75%,12021.000000,0.520747
max,47047.000000,0.714286


In [6]:
# hora y mes son ciclicas: 23h esta junto a 0h, diciembre junto a enero
hora_dec = df["hora"] + df["min"] / 60

df["hora_sin"] = np.sin(2 * np.pi * hora_dec / 24)
df["hora_cos"] = np.cos(2 * np.pi * hora_dec / 24)

# hora desconocida -> (0, 0), el centro del circulo: sin direccion horaria.
# hora_conocida ya marca exactamente esas filas, asi no inventamos una hora falsa
df[["hora_sin", "hora_cos"]] = df[["hora_sin", "hora_cos"]].fillna(0)

df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

df = df.drop(columns=["hora", "min", "mes"])

df[["hora_sin", "hora_cos", "mes_sin", "mes_cos", "hora_conocida"]].head()

,hora_sin,hora_cos,mes_sin,mes_cos,hora_conocida
0,0.0,1.0,1.0,6.123234e-17,1
1,0.0,1.0,1.0,6.123234e-17,1
2,0.0,1.0,1.0,6.123234e-17,1
3,0.0,1.0,1.0,6.123234e-17,1
4,0.0,1.0,0.5,8.660254e-01,1


In [7]:
# one-hot de municipio y dia_semana.
# ponemos la categoria de referencia al principio para que drop_first la quite:
# los coeficientes se leen contra Guadalajara y contra el lunes
dias = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]
df["dia_semana"] = pd.Categorical(df["dia_semana"].map(dict(enumerate(dias))), categories=dias)

munis = ["GUADALAJARA"] + sorted(set(df["municipio"]) - {"GUADALAJARA"})
df["municipio"] = pd.Categorical(df["municipio"], categories=munis)

# es_fin_semana = sabado + domingo, o sea es exactamente combinacion lineal de las
# dummies: dejarla haria singular la matriz de diseno. El one-hot ya la contiene.
df = df.drop(columns="es_fin_semana")

df = pd.get_dummies(df, columns=["municipio", "dia_semana"], drop_first=True, dtype=int)

nuevas = [c for c in df.columns if c.startswith(("municipio_", "dia_semana_"))]
print(len(nuevas), "columnas nuevas:")
for c in nuevas:
    print("  ", c)
df.head()

14 columnas nuevas:
   municipio_EL SALTO
   municipio_IXTLAHUACAN DE LOS MEMBRILLOS
   municipio_JUANACATLAN
   municipio_SAN PEDRO TLAQUEPAQUE
   municipio_TLAJOMULCO DE ZUÑIGA
   municipio_TONALA
   municipio_ZAPOPAN
   municipio_ZAPOTLANEJO
   dia_semana_martes
   dia_semana_miercoles
   dia_semana_jueves
   dia_semana_viernes
   dia_semana_sabado
   dia_semana_domingo


,delito,bien_afectado,colonia,año,hora_conocida,poblacion_colonia,razon_femenina,hora_sin,hora_cos,mes_sin,...,municipio_TLAJOMULCO DE ZUÑIGA,municipio_TONALA,municipio_ZAPOPAN,municipio_ZAPOTLANEJO,dia_semana_martes,dia_semana_miercoles,dia_semana_jueves,dia_semana_viernes,dia_semana_sabado,dia_semana_domingo
0,Abuso sexual infantil,La libertad y la seguridad sexual,JARDINES DEL BOSQUE,2020,1,6068.0,0.527851,0.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,1
1,Abuso sexual infantil,La libertad y la seguridad sexual,BALCONES DE SANTA CRUZ,2020,1,225.0,0.524444,0.0,1.0,1.0,...,0,1,0,0,0,1,0,0,0,0
2,Abuso sexual infantil,La libertad y la seguridad sexual,EL BETHEL,2020,1,8987.0,0.507511,0.0,1.0,1.0,...,0,0,0,0,0,0,0,0,1,0
3,Abuso sexual infantil,La libertad y la seguridad sexual,MIRAVALLE,2020,1,24199.0,0.523699,0.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,0
4,Abuso sexual infantil,La libertad y la seguridad sexual,SANTA MARIA,2020,1,10550.0,0.513081,0.0,1.0,0.5,...,0,0,0,0,1,0,0,0,0,0


In [8]:
# ============================================================
#  INDICE DE SEGURIDAD (SI) PARA OPTIMIZACION DE RUTAS
#
#      SI = N [ alpha * Rv + (1 - alpha) * Rp ]
#
#  Rv, Rp : razon de crimen violento / contra la propiedad en el periodo T
#  alpha  : peso de la severidad del crimen
#  N      : escala el resultado al rango 0-100
#
#  Trabajamos sobre datos-2026-09-09.csv y no sobre Crime_Population.csv porque
#  es el unico archivo que conserva las coordenadas (x, y) de cada delito, que
#  son indispensables para asignar un costo a cada arista del grafo de calles.
# ============================================================
geo = pd.read_csv("datos-2026-09-09.csv", encoding="utf-8-sig")
geo["fecha_dt"] = pd.to_datetime(geo["fecha"], dayfirst=True, errors="coerce")

print("registros:", len(geo))
print("sin coordenadas:", int(geo.x.isna().sum()))
print("rango:", geo.fecha_dt.min().date(), "a", geo.fecha_dt.max().date())
geo.head(3)

registros: 306152
sin coordenadas: 5772
rango: 2020-01-01 a 2026-07-31


,fecha,delito,x,y,colonia,municipio,clave_mun,hora,bien_afectado,zona_geografica,fecha_dt
0,04/02/2020,Abuso sexual infantil,-103.471379,20.775161,VILLA FONTANA DIAMANTE,ZAPOPAN,120,N.D.,La libertad y la seguridad sexual,AMG,2020-02-04
1,05/02/2020,Abuso sexual infantil,-103.473976,20.642161,LOMAS DE LA PRIMAVERA,ZAPOPAN,120,N.D.,La libertad y la seguridad sexual,AMG,2020-02-05
2,05/02/2020,Abuso sexual infantil,-103.365352,20.497948,CHULAVISTA E1,TLAJOMULCO DE ZUÑIGA,97,N.D.,La libertad y la seguridad sexual,AMG,2020-02-05


In [9]:
def parse_hora(col):
    """La columna 'hora' del archivo original mezcla cinco formatos:

         'HH:MM' / 'HH:MM:SS'   87.1%  -> lectura directa
         '0.177083333'           2.8%  -> fraccion de dia de Excel (x 24)
         '4.17E-02'              0.2%  -> la misma fraccion, notacion cientifica
         '19.05' / '1515'        2.8%  -> HH.MM y HHMM pegados
         'N.D.'                  9.9%  -> hora no registrada -> NaN

    Devuelve horas decimales en [0, 24). Sin esto, ~13% de los registros
    caen en la banda horaria equivocada.
    """
    s = col.astype(str).str.strip()
    out = pd.Series(np.nan, index=s.index, dtype=float)

    # 1) formato de reloj
    reloj = s.str.match(r"^\d{1,2}:\d{2}(:\d{2})?$")
    hm = s[reloj].str.split(":", expand=True).astype(float)
    out[reloj] = hm[0] + hm[1] / 60

    # 2) el resto, cuando es interpretable como numero
    num = pd.to_numeric(s.where(~reloj), errors="coerce")

    frac = num.between(0, 1, inclusive="neither")        # fraccion de dia
    out[frac] = num[frac] * 24

    entera = num.between(1, 24) & (num % 1 == 0)         # 18 -> 18:00
    out[entera] = num[entera]

    decimal = num.between(1, 24) & (num % 1 != 0)        # 19.05 -> 19:05
    out[decimal] = num[decimal].astype(int) + (num[decimal] % 1) * 100 / 60

    hhmm = num > 24                                      # 1515 -> 15:15
    out[hhmm] = (num[hhmm] // 100) + (num[hhmm] % 100) / 60

    out[num == 0] = 0.0
    return out.where(out.between(0, 24)) % 24            # 24:00 -> 0


geo["hora_dec"] = parse_hora(geo["hora"])

print("hora recuperada:", f"{geo.hora_dec.notna().mean():.1%}",
      "| sin hora:", int(geo.hora_dec.isna().sum()))
print(geo.hora_dec.describe().round(2).to_string())

hora recuperada: 90.1% | sin hora: 30281
count    275871.00
mean         13.79
std           6.39
min           0.00
25%           9.00
50%          14.17
75%          19.50
max          23.98


In [10]:
# Clasificacion de los 16 delitos segun su relevancia para quien transita por la
# via publica, que es lo que el SI debe medir.
#
# Se excluye el 41.4% de los registros y la razon importa: violencia familiar
# (64,016 casos, el delito mas frecuente del dataset) y abuso sexual infantil
# ocurren DENTRO del hogar; robo a negocio, casa habitacion y bancos tienen como
# victima al inmueble. Incluirlos convertiria el indice en un mapa de incidencia
# delictiva general, que penaliza zonas residenciales densas en lugar de las
# calles efectivamente peligrosas para el viajero.
VIOLENTO = ["Lesiones dolosas", "Homicidio doloso", "Violacion", "Feminicidio"]

PROPIEDAD = ["Robo a persona", "Robo a vehiculos particulares", "Robo de motocicleta",
             "Robo a int de vehiculos", "Robo de autopartes", "Robo a cuentahabientes",
             "Robo a carga pesada"]

geo["clase_ruta"] = np.select(
    [geo.delito.isin(VIOLENTO), geo.delito.isin(PROPIEDAD)],
    ["violento", "propiedad"],
    default="excluido")

print(geo.clase_ruta.value_counts().to_string())
print()
print((geo.clase_ruta.value_counts(normalize=True) * 100).round(1).to_string())

# base del indice: relevante para ruta + con coordenadas + con hora
si_df = geo[(geo.clase_ruta != "excluido") & geo.x.notna() & geo.y.notna()
            & geo.hora_dec.notna() & geo.fecha_dt.notna()].copy()

print("\nregistros usables para el SI:", len(si_df), f"({len(si_df) / len(geo):.1%} del total)")
print("razon propiedad / violento:",
      round((si_df.clase_ruta == "propiedad").sum() / (si_df.clase_ruta == "violento").sum(), 2))

clase_ruta
propiedad    134022
excluido     126706
violento      45424



clase_ruta
propiedad    43.8
excluido     41.4
violento     14.8

registros usables para el SI: 171101 (55.9% del total)
razon propiedad / violento: 3.07


In [11]:
# Rejilla espacial y bandas horarias.
#
# La rejilla cruda NO alcanza para calcular Rv y Rp como conteos directos: a
# 500 m x 4 bandas la mediana es de 7 delitos por celda-banda y solo el 43%
# llega a 10, asi que las razones estarian dominadas por el azar. Por eso
# suavizamos con un kernel gaussiano.
#
# BW = 400 m es el ancho elegido tras medir estabilidad temporal contra
# resolucion espacial:   200 m -> 0.835    400 m -> 0.926    800 m -> 0.972
# (Spearman entre el SI de 2020-2022 y el de 2023-2026). Mas suavizado es mas
# estable pero difumina la diferencia entre calles vecinas, que es justo lo que
# el optimizador necesita para elegir entre rutas alternativas.
from sklearn.neighbors import BallTree

METROS_CELDA = 500
BW = 400            # ancho del kernel gaussiano, en metros
N_BANDAS = 4        # bandas de 6 h: el periodo T de la formula
ALPHA = 0.7         # severidad: los violentos son 1 de cada 4 registros usables
R_TIERRA = 6371000.0

LAT_REF = np.radians(si_df.y.mean())
GRADO_X = METROS_CELDA / (111320 * np.cos(LAT_REF))
GRADO_Y = METROS_CELDA / 110540

si_df["banda"] = (si_df.hora_dec // (24 / N_BANDAS)).astype(int)

X0, Y0 = si_df.x.min(), si_df.y.min()
si_df["cx"] = ((si_df.x - X0) / GRADO_X).astype(int)
si_df["cy"] = ((si_df.y - Y0) / GRADO_Y).astype(int)

celdas = si_df.groupby(["cx", "cy"]).size().reset_index(name="delitos")
celdas["x"] = X0 + (celdas.cx + 0.5) * GRADO_X      # centro de la celda
celdas["y"] = Y0 + (celdas.cy + 0.5) * GRADO_Y

print("celdas de", METROS_CELDA, "m con al menos un delito:", len(celdas))
print("delitos por celda-banda:",
      si_df.groupby(["cx", "cy", "banda"]).size().median(), "(mediana)")

celdas de 500 m con al menos un delito: 4034
delitos por celda-banda: 7.0 (mediana)


In [12]:
def densidad(puntos, destino, bw=BW):
    """Densidad gaussiana de 'puntos' evaluada en 'destino' (lat, lon en radianes).

    Equivale a contar los delitos cercanos ponderando cada uno por su distancia,
    de modo que un delito a 50 m pesa mucho mas que uno a 1 km. Se truncan los
    vecinos a 3*bw, donde el peso gaussiano ya es despreciable.
    """
    if len(puntos) == 0:
        return np.zeros(len(destino))

    arbol = BallTree(np.radians(puntos[["y", "x"]].values), metric="haversine")
    dens = np.zeros(len(destino))
    py, px = puntos.y.values, puntos.x.values

    for i, ix in enumerate(arbol.query_radius(destino, r=3 * bw / R_TIERRA)):
        if len(ix) == 0:
            continue
        dlat = (np.radians(py[ix]) - destino[i, 0]) * R_TIERRA
        dlon = (np.radians(px[ix]) - destino[i, 1]) * R_TIERRA * np.cos(destino[i, 0])
        dens[i] = np.exp(-(dlat ** 2 + dlon ** 2) / (2 * bw ** 2)).sum()
    return dens


def indice_seguridad(datos, celdas, alpha=ALPHA, bw=BW):
    """SI = N [alpha*Rv + (1-alpha)*Rp] por celda y banda horaria.

    Rv y Rp quedan en delitos-equivalentes por anio, y N normaliza cada
    componente por su maximo antes de escalar a 0-100.
    """
    destino = np.radians(celdas[["y", "x"]].values)
    anios = datos.fecha_dt.dt.year.nunique()
    partes = []

    for b in range(N_BANDAS):
        sub = datos[datos.banda == b]
        Rv = densidad(sub[sub.clase_ruta == "violento"], destino, bw) / anios
        Rp = densidad(sub[sub.clase_ruta == "propiedad"], destino, bw) / anios
        si = 100 * (alpha * Rv / max(Rv.max(), 1e-9)
                    + (1 - alpha) * Rp / max(Rp.max(), 1e-9))
        partes.append(celdas.assign(banda=b, Rv=Rv, Rp=Rp, SI=si))

    return pd.concat(partes, ignore_index=True)


# recorre 4 bandas x ~4,000 celdas (un par de segundos)
si_celdas = indice_seguridad(si_df, celdas)

print("filas (celda x banda):", len(si_celdas))
print(si_celdas.groupby("banda").SI.agg(["mean", "median", "max"]).round(2).to_string())

filas (celda x banda): 16136
        mean  median     max
banda                       
0       9.94    5.77   82.76
1      11.67    7.50   76.19
2       9.26    5.98  100.00
3      12.50    7.98   77.11


In [13]:
# Validacion del indice. Dos preguntas, ambas necesarias antes de usar el SI
# como costo de arista:
#
# (1) ESTABILIDAD: el SI calculado con 2020-2022 debe ordenar las celdas igual
#     que el calculado con 2023-2026. Si no lo hace, el indice es ruido y no
#     sirve para rutear hoy con datos historicos.
# (2) CONTRASTE ENTRE BANDAS: mide cuanto aporta realmente el periodo T de la
#     formula. Si las bandas correlacionan ~1, la hora no cambia el ranking.
from scipy.stats import spearmanr

ETIQUETAS = {0: "00-06h", 1: "06-12h", 2: "12-18h", 3: "18-24h"}

si_previo = indice_seguridad(si_df[si_df.fecha_dt.dt.year <= 2022], celdas)
si_reciente = indice_seguridad(si_df[si_df.fecha_dt.dt.year >= 2023], celdas)

print("estabilidad temporal (2020-2022 vs 2023-2026):")
for b in range(N_BANDAS):
    a = si_previo.loc[si_previo.banda == b, "SI"].values
    c = si_reciente.loc[si_reciente.banda == b, "SI"].values
    print(f"   {ETIQUETAS[b]}   Spearman {spearmanr(a, c).statistic:.3f}")

print("\ncorrelacion entre bandas (cerca de 1 => la hora casi no altera el ranking):")
for i in range(N_BANDAS):
    for j in range(i + 1, N_BANDAS):
        a = si_celdas.loc[si_celdas.banda == i, "SI"].values
        c = si_celdas.loc[si_celdas.banda == j, "SI"].values
        print(f"   {ETIQUETAS[i]} vs {ETIQUETAS[j]}   {spearmanr(a, c).statistic:.3f}")

estabilidad temporal (2020-2022 vs 2023-2026):
   00-06h   Spearman 0.900
   06-12h   Spearman 0.924
   12-18h   Spearman 0.932
   18-24h   Spearman 0.946

correlacion entre bandas (cerca de 1 => la hora casi no altera el ranking):
   00-06h vs 06-12h   0.938
   00-06h vs 12-18h   0.944
   00-06h vs 18-24h   0.952
   06-12h vs 12-18h   0.957
   06-12h vs 18-24h   0.949
   12-18h vs 18-24h   0.964


In [14]:
# Salida para el optimizador de rutas.
#
# IMPORTANTE para pesar una arista del grafo de calles:
#
#     costo_seguridad(arista) = SI(punto medio, banda) * longitud(arista)
#
# El SI es una densidad puntual, no un costo por recorrido. Si se usa el SI solo,
# el optimizador prefiere rutas con pocas aristas largas en lugar de rutas
# realmente mas seguras. Multiplicar por la longitud hace que una arista larga en
# zona moderada pueda costar mas que una corta en zona peligrosa.
salida = si_celdas[["cx", "cy", "x", "y", "banda", "delitos", "Rv", "Rp", "SI"]]
salida.to_csv("indice_seguridad_celdas.csv", index=False)
print("guardado: indice_seguridad_celdas.csv", salida.shape)


def si_en_punto(lon, lat, banda, tabla=si_celdas):
    """SI de la celda que contiene el punto dado.

    Devuelve 0.0 si la celda no tiene delitos registrados. Ojo: ese 0 significa
    'sin registros', que no es lo mismo que 'seguro' -- puede ser una zona sin
    cobertura. Para rutear conviene tratarlo como el SI mediano de la zona.
    """
    cx = int((lon - X0) / GRADO_X)
    cy = int((lat - Y0) / GRADO_Y)
    m = tabla[(tabla.cx == cx) & (tabla.cy == cy) & (tabla.banda == banda)]
    return float(m.SI.iloc[0]) if len(m) else 0.0


for nombre, lon, lat in [("Centro de Guadalajara", -103.3496, 20.6767),
                         ("Zapopan centro", -103.3840, 20.7214),
                         ("Tlajomulco centro", -103.4467, 20.4750)]:
    valores = [round(si_en_punto(lon, lat, b), 1) for b in range(N_BANDAS)]
    print(f"{nombre:24} SI por banda {valores}")

guardado: indice_seguridad_celdas.csv (16136, 9)
Centro de Guadalajara    SI por banda [61.4, 66.3, 89.4, 66.6]
Zapopan centro           SI por banda [8.7, 13.8, 10.5, 12.2]
Tlajomulco centro        SI por banda [29.5, 36.2, 38.9, 34.3]


In [15]:
# ============================================================
#  PREDICCION DE OCURRENCIA (panel colonia x mes)
#
#  Reemplaza el intento anterior (es_violento). Ese clasificaba el TIPO de un
#  delito ya ocurrido -- no tiene clase negativa, no puede predecir si va a
#  haber un delito o no. El paper en el que nos basamos (Deep Multi-view
#  Spatio-Temporal Network for Urban Crime Prediction, Salama et al.) formula
#  el problema como clasificacion binaria de OCURRENCIA por region y ventana
#  de tiempo: "does crime type k happen in community i, or not". Replicamos
#  esa formulacion -- no su arquitectura (CNN+LSTM+GAN+AdaBoost, que requiere
#  datos de Twitter, flujo de taxis y clima que no tenemos), sino su idea de
#  separar variables espaciales y temporales en dos vistas.
#
#  Panel completo colonia x (anio, mes), incluyendo ceros: eso es lo que le da
#  una clase negativa real (colonia-mes SIN delitos relevantes), a diferencia
#  de trabajar solo con las filas de delitos que ya ocurrieron.
# ============================================================
rel = geo[(geo.clase_ruta != "excluido") & geo.fecha_dt.notna() & geo.colonia.notna()
          & geo.x.notna() & geo.y.notna()].copy()
rel["anio"] = rel.fecha_dt.dt.year
rel["mes_num"] = rel.fecha_dt.dt.month
rel["periodo"] = rel["anio"] * 100 + rel["mes_num"]

centroides = rel.groupby("colonia")[["x", "y"]].mean()

cp = pd.read_csv("Crime_Population.csv", encoding="utf-8")
attrs = cp.groupby("colonia").agg(municipio=("municipio", "first"),
                                   poblacion_colonia=("poblacion_colonia", "first")).reset_index()

colonias_ok = sorted(set(centroides.index) & set(attrs.colonia))
periodos_ok = sorted(rel.periodo.unique())
print("colonias utilizables:", len(colonias_ok), "| periodos:", len(periodos_ok),
      f"({periodos_ok[0]}-{periodos_ok[-1]})")

panel = pd.MultiIndex.from_product([colonias_ok, periodos_ok], names=["colonia", "periodo"]).to_frame(index=False)

conteo = rel[rel.colonia.isin(colonias_ok)].groupby(["colonia", "periodo"]).size().reset_index(name="n_delitos")
panel = panel.merge(conteo, on=["colonia", "periodo"], how="left")
panel["n_delitos"] = panel["n_delitos"].fillna(0).astype(int)
panel = panel.merge(attrs, on="colonia", how="left")
panel = panel.merge(centroides.rename(columns={"x": "cx", "y": "cy"}), on="colonia", how="left")
panel["anio"] = panel.periodo // 100
panel["mes_num"] = panel.periodo % 100

# target: hubo o no hubo al menos un delito relevante para ruta en esa colonia-mes
panel["ocurre"] = (panel.n_delitos > 0).astype(int)

print(f"\ntamano del panel: {len(panel)} filas (colonia x mes)")
print(f"celdas con 0 delitos: {(panel.n_delitos == 0).mean():.1%}  (= clase negativa real)")
print(f"prevalencia de ocurrencia: {panel.ocurre.mean():.3f}")

colonias utilizables: 1540 | periodos: 79 (202001-202607)



tamano del panel: 121660 filas (colonia x mes)
celdas con 0 delitos: 56.5%  (= clase negativa real)
prevalencia de ocurrencia: 0.435


In [16]:
# ============================================================
#  DOS VISTAS DE FEATURES, AMBAS USANDO SOLO INFORMACION PASADA
#
#  Vista TEMPORAL: autocorrelacion propia de la colonia (equivalente
#  simplificado al LSTM del paper sobre el historial de la propia region).
#
#  Vista ESPACIAL: promedio de la colonia-mes anterior entre las 10 colonias
#  vecinas mas cercanas (equivalente simplificado a su geo-matrix, que
#  regulariza con informacion de OTRAS regiones).
#
#  Las dos vistas usan unicamente datos de periodos anteriores al que se
#  predice -- nunca informacion de la propia colonia-mes que es el target --
#  para que el escenario sea de pronostico real y no de fuga de informacion.
# ============================================================
panel = panel.sort_values(["colonia", "periodo"])
g = panel.groupby("colonia")["n_delitos"]
panel["lag1"] = g.shift(1)
panel["roll3"] = g.shift(1).rolling(3).mean().reset_index(level=0, drop=True)
panel["mes_sin"] = np.sin(2 * np.pi * panel["mes_num"] / 12)
panel["mes_cos"] = np.cos(2 * np.pi * panel["mes_num"] / 12)

coords = centroides.loc[colonias_ok, ["x", "y"]].reset_index()
tree = BallTree(np.radians(coords[["y", "x"]].values), metric="haversine")
_, idx = tree.query(np.radians(coords[["y", "x"]].values), k=11)  # self + 10 vecinas
idx_vecinos = idx[:, 1:]

lag1_wide = panel.pivot(index="colonia", columns="periodo", values="lag1").loc[coords.colonia]
vecinos_lag1 = pd.DataFrame(
    np.nanmean(lag1_wide.values[idx_vecinos], axis=1),
    index=coords.colonia, columns=lag1_wide.columns
).stack().rename("vecinos_lag1").reset_index()
vecinos_lag1.columns = ["colonia", "periodo", "vecinos_lag1"]

panel = panel.merge(vecinos_lag1, on=["colonia", "periodo"], how="left")
panel["log_poblacion"] = np.log1p(panel["poblacion_colonia"])

usable = panel.dropna(subset=["lag1", "roll3", "vecinos_lag1", "poblacion_colonia"]).reset_index(drop=True)
print("filas usables (con historial de al menos 3 meses):", len(usable), "de", len(panel))
usable[["lag1", "roll3", "vecinos_lag1"]].describe().round(2)

filas usables (con historial de al menos 3 meses): 105412 de 121660


C:\Users\cvh44\AppData\Local\Temp\ipykernel_32804\593546634.py:29: RuntimeWarning: Mean of empty slice
  np.nanmean(lag1_wide.values[idx_vecinos], axis=1),


,lag1,roll3,vecinos_lag1
count,105412.00,105412.00,105412.00
mean,1.52,1.53,1.46
std,3.43,3.30,1.69
min,0.00,0.00,0.00
25%,0.00,0.00,0.40
50%,0.00,0.33,0.90
75%,2.00,1.67,1.90
max,121.00,124.67,24.10


In [17]:
# ============================================================
#  SPLIT CRONOLOGICO (no aleatorio): se entrena con el pasado y se prueba con
#  lo mas reciente, igual que el paper ("testing samples contain the more
#  recent data than training samples"). Un split aleatorio aqui filtraria
#  informacion del futuro hacia el entrenamiento.
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

periodos_usables = sorted(usable.periodo.unique())
n = len(periodos_usables)
corte_train = periodos_usables[int(n * 0.6)]
corte_val = periodos_usables[int(n * 0.8)]

train = usable[usable.periodo < corte_train].copy()
val = usable[(usable.periodo >= corte_train) & (usable.periodo < corte_val)].copy()
test = usable[usable.periodo >= corte_val].copy()
print(f"train: {len(train)} (hasta {corte_train})  |  val: {len(val)}  |  test: {len(test)} (desde {corte_val})")

munis = ["GUADALAJARA"] + sorted(set(usable.municipio) - {"GUADALAJARA"})
for parte in (train, val, test):
    parte["municipio"] = pd.Categorical(parte["municipio"], categories=munis)

def vista_temporal(p):
    return p[["lag1", "roll3", "mes_sin", "mes_cos"]].reset_index(drop=True)

def vista_espacial(p):
    return pd.concat([
        p[["vecinos_lag1", "log_poblacion"]].reset_index(drop=True),
        pd.get_dummies(p["municipio"], prefix="mun", drop_first=True).reset_index(drop=True),
    ], axis=1)

y_tr, y_val, y_te = train.ocurre.values, val.ocurre.values, test.ocurre.values

clf_temporal = LogisticRegression(max_iter=2000).fit(vista_temporal(train), y_tr)
clf_espacial = LogisticRegression(max_iter=2000).fit(vista_espacial(train), y_tr)

p_temp_val = clf_temporal.predict_proba(vista_temporal(val))[:, 1]
p_esp_val = clf_espacial.predict_proba(vista_espacial(val))[:, 1]
p_temp_te = clf_temporal.predict_proba(vista_temporal(test))[:, 1]
p_esp_te = clf_espacial.predict_proba(vista_espacial(test))[:, 1]

print("\nAUC vista temporal sola (test):", round(roc_auc_score(y_te, p_temp_te), 3))
print("AUC vista espacial sola (test):", round(roc_auc_score(y_te, p_esp_te), 3))

# Ws se elige en val (nunca en test), con Ws + Wt = 1
grid = [(Ws, roc_auc_score(y_val, Ws * p_esp_val + (1 - Ws) * p_temp_val))
        for Ws in np.arange(0.0, 1.01, 0.05)]
Ws_opt = max(grid, key=lambda t: t[1])[0]
p_final_te = Ws_opt * p_esp_te + (1 - Ws_opt) * p_temp_te

print(f"\nWs optimo (elegido en val): {Ws_opt:.2f}  |  Wt = {1 - Ws_opt:.2f}")
print("AUC combinado, dos vistas (test):", round(roc_auc_score(y_te, p_final_te), 3))
print("accuracy combinado (test, umbral 0.5):", round(accuracy_score(y_te, p_final_te > 0.5), 3))
print("base rate (clase mayoritaria, test):", round(max(y_te.mean(), 1 - y_te.mean()), 3))

# referencia: una sola logistica con todas las variables juntas, sin separar en vistas
Xc_tr = pd.concat([vista_temporal(train), vista_espacial(train)], axis=1)
Xc_te = pd.concat([vista_temporal(test), vista_espacial(test)], axis=1)
p_unico = LogisticRegression(max_iter=2000).fit(Xc_tr, y_tr).predict_proba(Xc_te)[:, 1]
print("\n(referencia) logistica unica con todas las variables juntas, AUC test:",
      round(roc_auc_score(y_te, p_unico), 3))

train: 62415 (hasta 202401)  |  val: 20805  |  test: 22192 (desde 202504)



AUC vista temporal sola (test): 0.833
AUC vista espacial sola (test): 0.83



Ws optimo (elegido en val): 0.35  |  Wt = 0.65
AUC combinado, dos vistas (test): 0.86
accuracy combinado (test, umbral 0.5): 0.786
base rate (clase mayoritaria, test): 0.574



(referencia) logistica unica con todas las variables juntas, AUC test: 0.859
